# ABG INSURANCE — PREDICTIVE PRICING NOTEBOOK

## Dormitory Renters Insurance — Case Competition Version

**Workflow:** data QA → exposure aggregation → development/holdout split → frequency modeling → severity modeling → holdout validation → development-only rating bases → Liability credibility adjustment → coverage-specific technical premium → policy-level pricing → renewal framework → monitoring and scenario analysis.

### Important governance principles
- The official holdout is reserved for final validation and is never used to construct rating bases.
- Gender is excluded from pricing and retained for monitoring only.
- Liability uses credibility because of limited claim volume and material holdout underprediction.
- The 20% variable expense and 10% profit/contingency assumptions are illustrative, not ABG facts.
- Final prices are technical indications subject to business, legal/regulatory, and independent validation review.


In [2]:
# -----------------------------
# 0. IMPORT PACKAGES
# -----------------------------
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression, GammaRegressor
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    RandomForestRegressor,
    GradientBoostingRegressor,
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
    log_loss,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_curve,
    precision_recall_curve,
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

from scipy.stats import norm, t


In [3]:
# 1. USER SETTINGS

# Credibility assumptions for low-volume Liability.
# These are modeling assumptions, not case facts.
CREDIBILITY_CONFIDENCE = 0.95
CREDIBILITY_RELATIVE_ERROR = 0.10


In [4]:
# CSV and Excel are both supported.
DATA_PATH = Path.cwd() / "06 - CAS Predictive Modeling Case Competition- Dataset.csv"

# Where to save outputs.
OUTPUT_DIR = Path("abg_final_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Pricing assumptions.
# These are illustrative because the case does not provide actual ABG expenses.
FIXED_EXPENSE_PER_COVERAGE = 0.00
VARIABLE_EXPENSE_RATIO = 0.20
PROFIT_AND_CONTINGENCY_RATIO = 0.10

# Optional minimum/maximum technical premium safeguards.
MIN_PREMIUM = 0.00
MAX_PREMIUM = None

# Variables excluded from the pricing model.
# "gender" is retained in the data for monitoring but is not used in pricing.
ALWAYS_EXCLUDE = [
    "student_id",
    "name",
    "claim_id",
    "amount",
    "has_claim",
    "claim_count",
    "training_holdout",
    "gender",
]

# Known policy limits from the case.
# The ALE limit is not supplied, so no cap is imposed here.
COVERAGE_LIMITS = {
    "Personal Property Protection": 10000,
    "Personal Property": 10000,
    "Liability Protection": 500000,
    "Liability": 500000,
    "Guest Medical Protection": 150000,
    "Guest Medical": 150000,
}

# Minimum number of positive claims needed to fit a severity model
# for a coverage. If lower, the code falls back to a pooled severity
# model rather than fitting an unstable coverage-specific model.
MIN_SEVERITY_CLAIMS = 40


In [5]:
# 2. HELPER FUNCTIONS


In [6]:
def clean_column_name(col):
    """Convert column names to lowercase snake_case."""
    col = str(col).strip().lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    return col.strip("_")


def load_data(path):
    """Load CSV or Excel input."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"\nData file not found: {path.resolve()}\n"
            "Update DATA_PATH at the top of the script."
        )

    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    elif path.suffix.lower() in [".xlsx", ".xls"]:
        df = pd.read_excel(path)
    else:
        raise ValueError("DATA_PATH must point to a CSV, XLSX, or XLS file.")

    df.columns = [clean_column_name(c) for c in df.columns]
    return df


def normalize_yes_no(series):
    """Standardize common yes/no representations where possible."""
    if series.dtype != "object":
        return series

    mapping = {
        "yes": "Yes", "y": "Yes", "true": "Yes", "1": "Yes",
        "no": "No", "n": "No", "false": "No", "0": "No",
    }
    s = series.astype(str).str.strip()
    lower = s.str.lower()
    return lower.map(mapping).fillna(s)


def infer_train_holdout(series):
    """
    Return Boolean masks for official training and holdout data.
    Handles labels such as train/training and holdout/test/validation.
    """
    s = series.astype(str).str.strip().str.lower()

    train_terms = {"train", "training", "model", "development", "dev", "0"}
    hold_terms = {"holdout", "test", "validation", "valid", "val", "1"}

    train_mask = s.isin(train_terms)
    hold_mask = s.isin(hold_terms)

    # If labels do not match the common names, use the two unique values.
    if (train_mask.sum() == 0 or hold_mask.sum() == 0) and s.nunique() == 2:
        print("\nWARNING: Could not identify training/holdout labels by name.")
        print("Observed values:", s.value_counts().to_dict())
        values = list(s.value_counts().index)
        # Assume the larger group is training.
        train_value = s.value_counts().idxmax()
        hold_value = [x for x in values if x != train_value][0]
        print(f"Assuming '{train_value}' = training and '{hold_value}' = holdout.")
        train_mask = s.eq(train_value)
        hold_mask = s.eq(hold_value)

    if train_mask.sum() == 0 or hold_mask.sum() == 0:
        raise ValueError(
            "Could not determine training and holdout rows from training_holdout."
        )

    return train_mask, hold_mask


def safe_auc(y_true, p):
    """AUC can fail if validation data contains only one class."""
    if pd.Series(y_true).nunique() < 2:
        return np.nan
    return roc_auc_score(y_true, p)


def frequency_metrics(y_true, p, threshold=0.50):
    pred = (np.asarray(p) >= threshold).astype(int)

    return {
        "ROC_AUC": safe_auc(y_true, p),
        "PR_AUC": average_precision_score(y_true, p),
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "Brier": brier_score_loss(y_true, p),
        "LogLoss": log_loss(y_true, np.clip(p, 1e-8, 1 - 1e-8)),
        "Observed_Frequency": np.mean(y_true),
        "Mean_Predicted_Probability": np.mean(p),
    }


def severity_metrics(y_true, pred):
    pred = np.maximum(np.asarray(pred), 0)
    return {
        "MAE": mean_absolute_error(y_true, pred),
        "RMSE": mean_squared_error(y_true, pred) ** 0.5,
        "R2": r2_score(y_true, pred),
        "Actual_Mean": np.mean(y_true),
        "Predicted_Mean": np.mean(pred),
    }


def make_group_split(data, test_size=0.25):
    """
    Grouped split by student_id to prevent one student's different
    coverages from appearing in both fitting and validation partitions.
    """
    gss = GroupShuffleSplit(
        n_splits=1,
        test_size=test_size,
        random_state=RANDOM_STATE,
    )

    train_idx, val_idx = next(
        gss.split(data, groups=data["student_id"])
    )
    return data.iloc[train_idx].copy(), data.iloc[val_idx].copy()


def identify_features(data):
    """Return candidate predictors and their numeric/categorical types."""
    candidate_features = [
        c for c in data.columns
        if c not in ALWAYS_EXCLUDE
    ]

    numeric_features = [
        c for c in candidate_features
        if pd.api.types.is_numeric_dtype(data[c])
    ]
    categorical_features = [
        c for c in candidate_features
        if c not in numeric_features
    ]

    return candidate_features, numeric_features, categorical_features


def make_preprocessor(numeric_features, categorical_features):
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    return ColumnTransformer([
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features),
    ])


def make_frequency_candidates(numeric_features, categorical_features):
    """
    Candidate frequency models.

    Balanced Logistic Regression is calibrated because class weights
    change the fitted probability scale. Calibration makes the resulting
    probabilities more appropriate for expected-loss pricing.
    """
    preprocessor = make_preprocessor(
        numeric_features, categorical_features
    )

    logistic = Pipeline([
        ("prep", clone(preprocessor)),
        ("model", LogisticRegression(
            max_iter=3000,
            solver="liblinear",
            random_state=RANDOM_STATE
        )),
    ])

    balanced_logistic_raw = Pipeline([
        ("prep", clone(preprocessor)),
        ("model", LogisticRegression(
            max_iter=3000,
            solver="liblinear",
            class_weight="balanced",
            random_state=RANDOM_STATE
        )),
    ])

    # Calibrate the weighted logistic model.
    balanced_logistic_calibrated = CalibratedClassifierCV(
        estimator=balanced_logistic_raw,
        method="sigmoid",
        cv=5,
    )

    rf = Pipeline([
        ("prep", clone(preprocessor)),
        ("model", RandomForestClassifier(
            n_estimators=200,
            min_samples_leaf=20,
            max_features="sqrt",
            class_weight=None,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])

    gb = Pipeline([
        ("prep", clone(preprocessor)),
        ("model", GradientBoostingClassifier(
            n_estimators=150,
            learning_rate=0.03,
            max_depth=2,
            min_samples_leaf=20,
            random_state=RANDOM_STATE,
        )),
    ])

    return {
        "Logistic": logistic,
        "Balanced_Logistic_Calibrated": balanced_logistic_calibrated,
        "Random_Forest": rf,
        "Gradient_Boosting": gb,
    }


def make_severity_candidates(numeric_features, categorical_features):
    preprocessor = make_preprocessor(
        numeric_features, categorical_features
    )

    gamma = Pipeline([
        ("prep", clone(preprocessor)),
        ("model", GammaRegressor(
            alpha=0.1,
            max_iter=3000,
        )),
    ])

    rf = Pipeline([
        ("prep", clone(preprocessor)),
        ("model", RandomForestRegressor(
            n_estimators=200,
            min_samples_leaf=10,
            max_features="sqrt",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])

    gb = Pipeline([
        ("prep", clone(preprocessor)),
        ("model", GradientBoostingRegressor(
            n_estimators=180,
            learning_rate=0.03,
            max_depth=2,
            min_samples_leaf=10,
            loss="huber",
            random_state=RANDOM_STATE,
        )),
    ])

    return {
        "Gamma_GLM": gamma,
        "Random_Forest": rf,
        "Gradient_Boosting": gb,
    }


def select_frequency_model(metrics_df):
    """
    Select a probability model for pricing.
    Primary criterion: Brier score.
    Safeguard: prefer models with useful discrimination (AUC >= 0.55)
    where available.

    The final choice should still be reviewed actuarially.
    """
    eligible = metrics_df[
        metrics_df["ROC_AUC"].fillna(0) >= 0.55
    ].copy()

    if eligible.empty:
        eligible = metrics_df.copy()

    return eligible.sort_values(
        ["Brier", "LogLoss", "ROC_AUC"],
        ascending=[True, True, False]
    ).iloc[0]["Model"]


def select_severity_model(metrics_df):
    """
    Severity selection: lowest RMSE, with MAE as tie-breaker.
    """
    return metrics_df.sort_values(
        ["RMSE", "MAE"],
        ascending=[True, True]
    ).iloc[0]["Model"]


def lift_table(y_true, p, n_bins=10):
    temp = pd.DataFrame({
        "actual": np.asarray(y_true),
        "predicted": np.asarray(p),
    })

    # Rank-based bins are more robust if probabilities repeat.
    temp["decile"] = pd.qcut(
        temp["predicted"].rank(method="first"),
        q=n_bins,
        labels=False,
        duplicates="drop",
    ) + 1

    out = (
        temp.groupby("decile", as_index=False)
        .agg(
            records=("actual", "size"),
            claims=("actual", "sum"),
            observed_frequency=("actual", "mean"),
            predicted_frequency=("predicted", "mean"),
        )
    )

    portfolio_freq = temp["actual"].mean()
    out["observed_lift"] = (
        out["observed_frequency"] / portfolio_freq
        if portfolio_freq > 0 else np.nan
    )

    return out.sort_values("decile", ascending=False)


def plot_frequency_diagnostics(
    y_true, p, coverage, model_name, output_dir
):
    safe_name = re.sub(r"[^A-Za-z0-9]+", "_", str(coverage)).strip("_")

    # ROC
    if pd.Series(y_true).nunique() > 1:
        fpr, tpr, _ = roc_curve(y_true, p)
        plt.figure(figsize=(7, 5))
        plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_true, p):.3f}")
        plt.plot([0, 1], [0, 1], linestyle="--")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(f"ROC - {coverage} - {model_name}")
        plt.legend()
        plt.tight_layout()
        plt.savefig(
            output_dir / f"roc_{safe_name}.png",
            dpi=160,
            bbox_inches="tight"
        )
        plt.close()

    # Precision-recall
    precision, recall, _ = precision_recall_curve(y_true, p)
    plt.figure(figsize=(7, 5))
    plt.plot(recall, precision)
    plt.axhline(np.mean(y_true), linestyle="--")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Precision-Recall - {coverage} - {model_name}")
    plt.tight_layout()
    plt.savefig(
        output_dir / f"pr_{safe_name}.png",
        dpi=160,
        bbox_inches="tight"
    )
    plt.close()

    # Calibration
    prob_true, prob_pred = calibration_curve(
        y_true, p, n_bins=10, strategy="quantile"
    )
    plt.figure(figsize=(7, 5))
    plt.plot(prob_pred, prob_true, marker="o")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("Mean Predicted Probability")
    plt.ylabel("Observed Claim Frequency")
    plt.title(f"Calibration - {coverage} - {model_name}")
    plt.tight_layout()
    plt.savefig(
        output_dir / f"calibration_{safe_name}.png",
        dpi=160,
        bbox_inches="tight"
    )
    plt.close()


def plot_severity_diagnostic(
    y_true, pred, coverage, model_name, output_dir
):
    safe_name = re.sub(r"[^A-Za-z0-9]+", "_", str(coverage)).strip("_")
    max_val = max(np.max(y_true), np.max(pred))

    plt.figure(figsize=(7, 5))
    plt.scatter(y_true, pred, alpha=0.4)
    plt.plot([0, max_val], [0, max_val], linestyle="--")
    plt.xlabel("Actual Claim Severity")
    plt.ylabel("Predicted Claim Severity")
    plt.title(f"Severity: Actual vs Predicted - {coverage} - {model_name}")
    plt.tight_layout()
    plt.savefig(
        output_dir / f"severity_actual_vs_pred_{safe_name}.png",
        dpi=160,
        bbox_inches="tight"
    )
    plt.close()


In [7]:
# 3. LOAD AND STANDARDIZE DATA


In [8]:
df_raw = load_data(DATA_PATH)
print("\nRAW DATA SHAPE:", df_raw.shape)
print("\nCOLUMNS:")
print(df_raw.columns.tolist())

required = {
    "student_id",
    "coverage",
    "amount",
    "training_holdout",
}
missing_required = required - set(df_raw.columns)

if missing_required:
    raise ValueError(
        f"Missing required columns: {sorted(missing_required)}"
    )

df = df_raw.copy()

# Clean selected text variables.
for col in [
    "coverage",
    "risk_tier",
    "training_holdout",
    "class",
    "study",
    "greek",
    "off_campus",
    "gender",
    "sprinklered",
]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

for col in ["greek", "off_campus", "sprinklered"]:
    if col in df.columns:
        df[col] = normalize_yes_no(df[col])

df["amount"] = pd.to_numeric(df["amount"], errors="coerce").fillna(0.0)
df["amount"] = df["amount"].clip(lower=0)

# Claim indicator.
df["has_claim"] = (df["amount"] > 0).astype(int)



RAW DATA SHAPE: (40071, 15)

COLUMNS:
['student_id', 'name', 'class', 'study', 'gpa', 'greek', 'off_campus', 'distance_to_campus', 'gender', 'sprinklered', 'coverage', 'claim_id', 'amount', 'risk_tier', 'training_holdout']


In [9]:
# 4. DATA AUDIT


In [10]:
print("\n" + "=" * 70)
print("DATA AUDIT")
print("=" * 70)

print("\nRows:", len(df))
print("Unique policyholders:", df["student_id"].nunique())
print("Unique coverages:", df["coverage"].nunique())
print("\nCoverage counts:")
print(df["coverage"].value_counts(dropna=False))

print("\nRows per student:")
print(df.groupby("student_id").size().value_counts().sort_index())

coverage_count = df.groupby("student_id")["coverage"].nunique()
print("\nNumber of unique coverages per student:")
print(coverage_count.value_counts().sort_index())

duplicate_student_coverage = df.duplicated(
    subset=["student_id", "coverage"]
).sum()

print(
    "\nRepeated student-coverage combinations:",
    duplicate_student_coverage
)

duplicate_detail = (
    df[df.duplicated(["student_id", "coverage"], keep=False)]
    .sort_values(["student_id", "coverage"])
)

duplicate_detail.to_csv(
    OUTPUT_DIR / "duplicate_student_coverage_records.csv",
    index=False
)

print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False))

print("\nOverall claim frequency:", df["has_claim"].mean())
print("Total claim amount:", df["amount"].sum())
print("Average loss per coverage record:", df["amount"].mean())



DATA AUDIT

Rows: 40071
Unique policyholders: 10000
Unique coverages: 4

Coverage counts:
coverage
Personal Property            10051
Additional Living Expense    10016
Guest Medical                10004
Liability                    10000
Name: count, dtype: int64

Rows per student:
4    9933
5      63
6       4
Name: count, dtype: int64

Number of unique coverages per student:
coverage
4    10000
Name: count, dtype: int64

Repeated student-coverage combinations: 71

Missing values:
student_id            0
name                  0
class                 0
study                 0
gpa                   0
greek                 0
off_campus            0
distance_to_campus    0
gender                0
sprinklered           0
coverage              0
claim_id              0
amount                0
risk_tier             0
training_holdout      0
has_claim             0
dtype: int64

Overall claim frequency: 0.045394424895809936
Total claim amount: 8514740.025596071
Average loss per coverage rec

In [11]:
# 5. RESOLVE REPEATED STUDENT-COVERAGE ROWS


In [12]:
#
# The data may contain multiple claims for the same student and coverage.
# Do NOT blindly drop them.
#
# We aggregate to one pricing record per student + coverage:
#   - amount = total loss for that student-coverage
#   - claim_count = number of positive claim rows
#   - claim_id = concatenated/count information is not used as predictor
#   - policy/risk characteristics = first observed value
#
# Before aggregation, we check whether policy characteristics are consistent
# inside each repeated student-coverage combination.

id_cols = ["student_id", "coverage"]

non_claim_cols = [
    c for c in df.columns
    if c not in ["amount", "claim_id", "has_claim"]
]

# Check within-group inconsistency in predictors.
consistency_rows = []
for col in non_claim_cols:
    if col in id_cols:
        continue

    nunique_by_group = (
        df.groupby(id_cols)[col]
        .nunique(dropna=False)
    )
    n_inconsistent = int((nunique_by_group > 1).sum())

    if n_inconsistent > 0:
        consistency_rows.append({
            "column": col,
            "inconsistent_student_coverage_groups": n_inconsistent
        })

consistency_report = pd.DataFrame(consistency_rows)

if not consistency_report.empty:
    print("\nWARNING: Some repeated groups have inconsistent policy values:")
    print(consistency_report)
    consistency_report.to_csv(
        OUTPUT_DIR / "duplicate_group_inconsistencies.csv",
        index=False
    )

# Aggregation dictionary.
agg_dict = {}

for col in df.columns:
    if col in id_cols:
        continue
    if col == "amount":
        agg_dict[col] = "sum"
    elif col == "has_claim":
        agg_dict[col] = "max"
    elif col == "claim_id":
        # not used for pricing; retain first for traceability
        agg_dict[col] = "first"
    else:
        agg_dict[col] = "first"

pricing_df = (
    df.groupby(id_cols, as_index=False)
      .agg(agg_dict)
)

# Explicit claim count based on original rows.
claim_counts = (
    df.assign(_positive=(df["amount"] > 0).astype(int))
      .groupby(id_cols, as_index=False)["_positive"]
      .sum()
      .rename(columns={"_positive": "claim_count"})
)

pricing_df = pricing_df.merge(
    claim_counts,
    on=id_cols,
    how="left"
)

pricing_df["has_claim"] = (
    pricing_df["amount"] > 0
).astype(int)

print("\nAFTER AGGREGATION")
print("Shape:", pricing_df.shape)
print(
    "Repeated student-coverage combinations:",
    pricing_df.duplicated(id_cols).sum()
)
print(
    "Unique policyholders:",
    pricing_df["student_id"].nunique()
)
print(
    "Unique coverages:",
    pricing_df["coverage"].nunique()
)



AFTER AGGREGATION
Shape: (40000, 17)
Repeated student-coverage combinations: 0
Unique policyholders: 10000
Unique coverages: 4


In [13]:
# 6. OFFICIAL TRAINING / HOLDOUT SPLIT


In [14]:
train_mask, holdout_mask = infer_train_holdout(
    pricing_df["training_holdout"]
)

train_df = pricing_df.loc[train_mask].copy()
holdout_df = pricing_df.loc[holdout_mask].copy()

print("\nTRAINING SHAPE:", train_df.shape)
print("HOLDOUT SHAPE:", holdout_df.shape)
print(
    "TRAINING CLAIM FREQUENCY:",
    train_df["has_claim"].mean()
)
print(
    "HOLDOUT CLAIM FREQUENCY:",
    holdout_df["has_claim"].mean()
)



TRAINING SHAPE: (31996, 17)
HOLDOUT SHAPE: (8004, 17)
TRAINING CLAIM FREQUENCY: 0.04403675459432429
HOLDOUT CLAIM FREQUENCY: 0.04235382308845577


In [15]:
# 7. EXPLORATORY DATA ANALYSIS


In [2]:
overall_summary = pd.DataFrame({
    "Metric": [
        "Policyholders",
        "Coverage records",
        "Positive claim records",
        "Claim frequency",
        "Total losses",
        "Average loss per coverage record",
        "Average severity conditional on claim",
    ],
    "Value": [
        pricing_df["student_id"].nunique(),
        len(pricing_df),
        pricing_df["has_claim"].sum(),
        pricing_df["has_claim"].mean(),
        pricing_df["amount"].sum(),
        pricing_df["amount"].mean(),
        pricing_df.loc[
            pricing_df["amount"] > 0, "amount"
        ].mean(),
    ],
})

coverage_summary = (
    pricing_df.groupby("coverage", as_index=False)
    .agg(
        records=("student_id", "size"),
        policyholders=("student_id", "nunique"),
        claims=("has_claim", "sum"),
        claim_frequency=("has_claim", "mean"),
        total_loss=("amount", "sum"),
        average_loss=("amount", "mean"),
        average_severity=(
            "amount",
            lambda x: x[x > 0].mean()
        ),
    )
)

if "risk_tier" in pricing_df.columns:
    risk_tier_summary = (
        pricing_df.groupby("risk_tier", as_index=False)
        .agg(
            records=("student_id", "size"),
            claims=("has_claim", "sum"),
            claim_frequency=("has_claim", "mean"),
            total_loss=("amount", "sum"),
            average_loss=("amount", "mean"),
            average_severity=(
                "amount",
                lambda x: x[x > 0].mean()
            ),
        )
    )
else:
    risk_tier_summary = pd.DataFrame()

print("\nOVERALL SUMMARY")
print(overall_summary)

print("\nCOVERAGE SUMMARY")
print(coverage_summary)

if not risk_tier_summary.empty:
    print("\nRISK TIER SUMMARY")
    print(risk_tier_summary)


# EDA charts
plt.figure(figsize=(9, 5))
coverage_summary = coverage_summary.sort_values(
    "claim_frequency", ascending=False
)
plt.bar(
    coverage_summary["coverage"],
    coverage_summary["claim_frequency"]
)
plt.xticks(rotation=30, ha="right")
plt.ylabel("Observed Claim Frequency")
plt.title("Claim Frequency by Coverage")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "eda_claim_frequency_by_coverage.png",
    dpi=160,
    bbox_inches="tight"
)
plt.close()

plt.figure(figsize=(9, 5))
coverage_summary2 = coverage_summary.sort_values(
    "average_severity", ascending=False
)
plt.bar(
    coverage_summary2["coverage"],
    coverage_summary2["average_severity"]
)
plt.xticks(rotation=30, ha="right")
plt.ylabel("Average Claim Severity")
plt.title("Average Positive Claim Severity by Coverage")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "eda_severity_by_coverage.png",
    dpi=160,
    bbox_inches="tight"
)
plt.close()

# Optional continuous-variable EDA.
for continuous_col in ["gpa", "distance_to_campus"]:
    if continuous_col in pricing_df.columns:
        temp = pricing_df[[continuous_col, "amount", "has_claim"]].copy()
        temp[continuous_col] = pd.to_numeric(
            temp[continuous_col], errors="coerce"
        )
        temp = temp.dropna(subset=[continuous_col])

        if not temp.empty:
            temp["band"] = pd.qcut(
                temp[continuous_col],
                q=5,
                duplicates="drop"
            )
            summary = (
                temp.groupby("band", observed=False)
                .agg(
                    records=("has_claim", "size"),
                    claim_frequency=("has_claim", "mean"),
                    average_loss=("amount", "mean"),
                )
                .reset_index()
            )
            summary.to_csv(
                OUTPUT_DIR / f"eda_{continuous_col}_bands.csv",
                index=False
            )


# ------------------------------------------------------------------
# LOSS RECONCILIATION AND RISK-TIER SOURCE
# ------------------------------------------------------------------
# Aggregating repeated student-coverage rows changes the number of records
# and the number of positive exposure records, but it does NOT change total
# loss dollars. The raw and aggregated totals are reconciled explicitly.
raw_total_loss = df_raw["amount"].sum()
aggregated_total_loss = pricing_df["amount"].sum()
loss_difference = raw_total_loss - aggregated_total_loss
raw_positive_claim_rows = int((df_raw["amount"] > 0).sum())
positive_claim_exposures = int((pricing_df["amount"] > 0).sum())

loss_reconciliation = pd.DataFrame({
    "Metric": [
        "Raw dataset rows",
        "Raw positive claim rows",
        "Aggregated student-coverage exposures",
        "Positive claim exposures after aggregation",
        "Raw total loss",
        "Aggregated total loss",
        "Loss difference",
    ],
    "Value": [
        len(df_raw), raw_positive_claim_rows, len(pricing_df),
        positive_claim_exposures, raw_total_loss, aggregated_total_loss, loss_difference
    ]
})
print("\nLOSS RECONCILIATION")
print(loss_reconciliation)

if "risk_tier" in pricing_df.columns:
    risk_tier_construction = (
        pricing_df.groupby("risk_tier", as_index=False)
        .agg(exposures=("student_id", "size"),
             share=("student_id", lambda s: len(s) / len(pricing_df)),
             claims=("has_claim", "sum"),
             total_loss=("amount", "sum"),
             pure_premium=("amount", "mean"))
        .sort_values("risk_tier")
    )
    print("\nRISK-TIER SOURCE AND DISTRIBUTION")
    print(risk_tier_construction)
else:
    risk_tier_construction = pd.DataFrame()


NameError: name 'pd' is not defined

In [17]:
# 8. FEATURE SET


In [18]:
features, numeric_features, categorical_features = identify_features(
    train_df
)

print("\nMODELING FEATURES:")
print(features)

print("\nNUMERIC FEATURES:")
print(numeric_features)

print("\nCATEGORICAL FEATURES:")
print(categorical_features)

if not features:
    raise ValueError("No predictors remain after exclusions.")



MODELING FEATURES:
['coverage', 'class', 'study', 'gpa', 'greek', 'off_campus', 'distance_to_campus', 'sprinklered', 'risk_tier']

NUMERIC FEATURES:
['gpa', 'distance_to_campus']

CATEGORICAL FEATURES:
['coverage', 'class', 'study', 'greek', 'off_campus', 'sprinklered', 'risk_tier']


In [19]:
# ------------------------------------------------------------------
# CREDIBILITY HELPERS
# ------------------------------------------------------------------
def limited_fluctuation_credibility(n, cv=None, confidence=CREDIBILITY_CONFIDENCE, relative_error=CREDIBILITY_RELATIVE_ERROR):
    """Return Z under a simple limited-fluctuation credibility standard."""
    z = norm.ppf(0.5 + confidence / 2)
    if cv is None or not np.isfinite(cv) or cv <= 0:
        # Poisson count approximation: relative error = z/sqrt(N).
        n_full = (z / relative_error) ** 2
    else:
        n_full = (z * cv / relative_error) ** 2
    Z = min(1.0, np.sqrt(max(n, 0) / n_full))
    return float(Z), float(n_full)

def wilson_interval(x, n, confidence=0.95):
    if n == 0:
        return np.nan, np.nan
    z = norm.ppf(0.5 + confidence / 2)
    p = x / n
    denom = 1 + z**2 / n
    center = (p + z**2/(2*n)) / denom
    half = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / denom
    return max(0.0, center-half), min(1.0, center+half)

def bootstrap_mean_ci(values, n_boot=2000, confidence=0.95, seed=RANDOM_STATE):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) < 2:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    means = np.empty(n_boot)
    for i in range(n_boot):
        means[i] = rng.choice(values, size=len(values), replace=True).mean()
    alpha = 1 - confidence
    return float(np.quantile(means, alpha/2)), float(np.quantile(means, 1-alpha/2))


In [20]:
# 9. COVERAGE-SPECIFIC FREQUENCY MODELS


In [21]:
frequency_results = []
frequency_models = {}
frequency_lift_tables = []
frequency_validation_predictions = []

coverages = sorted(train_df["coverage"].dropna().unique())

for coverage in coverages:
    print("\n" + "=" * 70)
    print("FREQUENCY MODEL:", coverage)
    print("=" * 70)

    cov_data = train_df[
        train_df["coverage"] == coverage
    ].copy()

    fit_df, val_df = make_group_split(
        cov_data, test_size=0.25
    )

    X_fit = fit_df[features]
    y_fit = fit_df["has_claim"].astype(int)

    X_val = val_df[features]
    y_val = val_df["has_claim"].astype(int)

    print(
        "Fit frequency:",
        y_fit.mean(),
        "| Validation frequency:",
        y_val.mean(),
        "| Validation claims:",
        y_val.sum()
    )

    candidates = make_frequency_candidates(
        numeric_features,
        categorical_features
    )

    local_results = []
    fitted_candidates = {}

    for name, model in candidates.items():
        try:
            model.fit(X_fit, y_fit)
            p_val = model.predict_proba(X_val)[:, 1]

            metrics = frequency_metrics(
                y_val,
                p_val,
                threshold=0.50
            )

            row = {
                "Coverage": coverage,
                "Model": name,
                **metrics,
            }
            frequency_results.append(row)
            local_results.append(row)
            fitted_candidates[name] = model

            print(
                f"{name}: "
                f"AUC={metrics['ROC_AUC']:.3f}, "
                f"PR_AUC={metrics['PR_AUC']:.3f}, "
                f"Brier={metrics['Brier']:.4f}, "
                f"Obs={metrics['Observed_Frequency']:.4f}, "
                f"Pred={metrics['Mean_Predicted_Probability']:.4f}"
            )

        except Exception as e:
            print(f"{name} FAILED: {e}")

    local_metrics = pd.DataFrame(local_results)

    if local_metrics.empty:
        raise RuntimeError(
            f"No frequency model fitted successfully for {coverage}."
        )

    best_name = select_frequency_model(local_metrics)
    print("SELECTED FREQUENCY MODEL:", best_name)

    # Refit selected model on all official training records for this coverage.
    final_model = make_frequency_candidates(
        numeric_features,
        categorical_features
    )[best_name]

    final_model.fit(
        cov_data[features],
        cov_data["has_claim"].astype(int)
    )

    frequency_models[coverage] = {
        "name": best_name,
        "model": final_model,
    }

    # Save validation diagnostics from the selected candidate.
    selected_val_model = fitted_candidates[best_name]
    p_val = selected_val_model.predict_proba(X_val)[:, 1]

    validation_pred = val_df[
        ["student_id", "coverage", "has_claim"]
    ].copy()
    validation_pred["predicted_probability"] = p_val
    validation_pred["model"] = best_name
    frequency_validation_predictions.append(validation_pred)

    lift = lift_table(y_val, p_val, n_bins=10)
    lift.insert(0, "Coverage", coverage)
    lift.insert(1, "Model", best_name)
    frequency_lift_tables.append(lift)

    plot_frequency_diagnostics(
        y_val,
        p_val,
        coverage,
        best_name,
        OUTPUT_DIR,
    )


frequency_metrics_df = pd.DataFrame(frequency_results)
frequency_validation_df = pd.concat(
    frequency_validation_predictions,
    ignore_index=True
)
lift_df = pd.concat(
    frequency_lift_tables,
    ignore_index=True
)



FREQUENCY MODEL: Additional Living Expense
Fit frequency: 0.050841806967827974 | Validation frequency: 0.0555 | Validation claims: 111
Logistic: AUC=0.599, PR_AUC=0.080, Brier=0.0521, Obs=0.0555, Pred=0.0513
Balanced_Logistic_Calibrated: AUC=0.599, PR_AUC=0.079, Brier=0.0521, Obs=0.0555, Pred=0.0511
Random_Forest: AUC=0.523, PR_AUC=0.058, Brier=0.0530, Obs=0.0555, Pred=0.0521
Gradient_Boosting: AUC=0.575, PR_AUC=0.075, Brier=0.0523, Obs=0.0555, Pred=0.0514
SELECTED FREQUENCY MODEL: Logistic

FREQUENCY MODEL: Guest Medical
Fit frequency: 0.022503750625104183 | Validation frequency: 0.024 | Validation claims: 48
Logistic: AUC=0.613, PR_AUC=0.053, Brier=0.0233, Obs=0.0240, Pred=0.0226
Balanced_Logistic_Calibrated: AUC=0.602, PR_AUC=0.053, Brier=0.0233, Obs=0.0240, Pred=0.0226
Random_Forest: AUC=0.619, PR_AUC=0.079, Brier=0.0233, Obs=0.0240, Pred=0.0226
Gradient_Boosting: AUC=0.634, PR_AUC=0.054, Brier=0.0233, Obs=0.0240, Pred=0.0231
SELECTED FREQUENCY MODEL: Random_Forest

FREQUENCY MODE

Balanced_Logistic_Calibrated: AUC=0.599, PR_AUC=0.079, Brier=0.0521, Obs=0.0555, Pred=0.0511


Random_Forest: AUC=0.527, PR_AUC=0.059, Brier=0.0530, Obs=0.0555, Pred=0.0519


Gradient_Boosting: AUC=0.569, PR_AUC=0.071, Brier=0.0524, Obs=0.0555, Pred=0.0515
SELECTED FREQUENCY MODEL: Logistic



FREQUENCY MODEL: Guest Medical
Fit frequency: 0.022503750625104183 | Validation frequency: 0.024 | Validation claims: 48
Logistic: AUC=0.613, PR_AUC=0.053, Brier=0.0233, Obs=0.0240, Pred=0.0226


...

Gradient_Boosting: AUC=0.560, PR_AUC=0.012, Brier=0.0085, Obs=0.0085, Pred=0.0082
SELECTED FREQUENCY MODEL: Balanced_Logistic_Calibrated



FREQUENCY MODEL: Personal Property
Fit frequency: 0.09334889148191365 | Validation frequency: 0.0915 | Validation claims: 183
Logistic: AUC=0.584, PR_AUC=0.144, Brier=0.0819, Obs=0.0915, Pred=0.0943


Balanced_Logistic_Calibrated: AUC=0.582, PR_AUC=0.146, Brier=0.0820, Obs=0.0915, Pred=0.0939


Random_Forest: AUC=0.592, PR_AUC=0.129, Brier=0.0826, Obs=0.0915, Pred=0.0941


Gradient_Boosting: AUC=0.579, PR_AUC=0.141, Brier=0.0821, Obs=0.0915, Pred=0.0944
SELECTED FREQUENCY MODEL: Logistic


In [22]:
# 10. COVERAGE-SPECIFIC SEVERITY MODELS


In [23]:
severity_results = []
severity_models = {}
severity_validation_predictions = []

# A pooled severity dataset is kept as a fallback if a coverage has too
# few positive claims to support a stable standalone severity model.
pooled_severity_df = train_df[
    train_df["amount"] > 0
].copy()

for coverage in coverages:
    print("\n" + "=" * 70)
    print("SEVERITY MODEL:", coverage)
    print("=" * 70)

    cov_severity = train_df[
        (train_df["coverage"] == coverage)
        & (train_df["amount"] > 0)
    ].copy()

    print("Positive claims:", len(cov_severity))

    use_pooled = len(cov_severity) < MIN_SEVERITY_CLAIMS

    if use_pooled:
        print(
            f"WARNING: Fewer than {MIN_SEVERITY_CLAIMS} claims. "
            "Using pooled severity data with coverage retained as a predictor."
        )
        modeling_data = pooled_severity_df.copy()
    else:
        modeling_data = cov_severity.copy()

    sev_fit, sev_val = make_group_split(
        modeling_data,
        test_size=0.25
    )

    X_fit = sev_fit[features]
    y_fit = sev_fit["amount"].astype(float)

    X_val = sev_val[features]
    y_val = sev_val["amount"].astype(float)

    candidates = make_severity_candidates(
        numeric_features,
        categorical_features
    )

    local_results = []
    fitted_candidates = {}

    for name, model in candidates.items():
        try:
            model.fit(X_fit, y_fit)
            pred = np.maximum(model.predict(X_val), 0)

            metrics = severity_metrics(y_val, pred)

            row = {
                "Coverage": coverage,
                "Model": name,
                "Pooled_Fallback": use_pooled,
                "Development_Positive_Claims": len(cov_severity),
                "Metric_Type": "Internal validation (25% group split of development claims)",
                **metrics,
            }
            severity_results.append(row)
            local_results.append(row)
            fitted_candidates[name] = model

            print(
                f"{name}: "
                f"MAE={metrics['MAE']:.2f}, "
                f"RMSE={metrics['RMSE']:.2f}, "
                f"R2={metrics['R2']:.3f}, "
                f"ActualMean={metrics['Actual_Mean']:.2f}, "
                f"PredMean={metrics['Predicted_Mean']:.2f}"
            )

        except Exception as e:
            print(f"{name} FAILED: {e}")

    local_metrics = pd.DataFrame(local_results)

    if local_metrics.empty:
        raise RuntimeError(
            f"No severity model fitted successfully for {coverage}."
        )

    best_name = select_severity_model(local_metrics)
    print("SELECTED SEVERITY MODEL:", best_name)

    final_model = make_severity_candidates(
        numeric_features,
        categorical_features
    )[best_name]

    # Final severity model is fitted to the chosen source data.
    final_model.fit(
        modeling_data[features],
        modeling_data["amount"].astype(float)
    )

    severity_models[coverage] = {
        "name": best_name,
        "model": final_model,
        "pooled_fallback": use_pooled,
    }

    # Diagnostic validation predictions.
    selected_val_model = fitted_candidates[best_name]
    pred = np.maximum(
        selected_val_model.predict(sev_val[features]),
        0
    )

    val_out = sev_val[
        ["student_id", "coverage", "amount"]
    ].copy()
    val_out["predicted_severity"] = pred
    val_out["model"] = best_name
    val_out["evaluated_for_coverage"] = coverage
    severity_validation_predictions.append(val_out)

    plot_severity_diagnostic(
        y_val,
        pred,
        coverage,
        best_name,
        OUTPUT_DIR,
    )


severity_metrics_df = pd.DataFrame(severity_results)
severity_validation_df = pd.concat(
    severity_validation_predictions,
    ignore_index=True
)



SEVERITY MODEL: Additional Living Expense
Positive claims: 416
Gamma_GLM: MAE=2720.55, RMSE=4229.14, R2=0.470, ActualMean=5911.73, PredMean=5322.89
Random_Forest: MAE=2937.83, RMSE=4630.93, R2=0.365, ActualMean=5911.73, PredMean=5491.81
Gradient_Boosting: MAE=2718.91, RMSE=4363.25, R2=0.436, ActualMean=5911.73, PredMean=5268.60
SELECTED SEVERITY MODEL: Gamma_GLM

SEVERITY MODEL: Guest Medical
Positive claims: 183
Gamma_GLM: MAE=11927.77, RMSE=18712.07, R2=0.345, ActualMean=19223.66, PredMean=17044.96
Random_Forest: MAE=13370.22, RMSE=21524.13, R2=0.134, ActualMean=19223.66, PredMean=17425.91
Gradient_Boosting: MAE=13308.82, RMSE=21783.98, R2=0.113, ActualMean=19223.66, PredMean=16949.52
SELECTED SEVERITY MODEL: Gamma_GLM

SEVERITY MODEL: Liability
Positive claims: 67
Gamma_GLM: MAE=8907.16, RMSE=13171.76, R2=-0.266, ActualMean=13210.28, PredMean=7308.67
Random_Forest: MAE=8559.00, RMSE=13265.40, R2=-0.284, ActualMean=13210.28, PredMean=6652.40
Gradient_Boosting: MAE=8486.60, RMSE=1340

Random_Forest: MAE=2923.17, RMSE=4597.05, R2=0.374, ActualMean=5911.73, PredMean=5499.27


Gradient_Boosting: MAE=2682.23, RMSE=4276.27, R2=0.458, ActualMean=5911.73, PredMean=5358.03
SELECTED SEVERITY MODEL: Gamma_GLM



SEVERITY MODEL: Guest Medical
Positive claims: 183
Gamma_GLM: MAE=11927.77, RMSE=18712.07, R2=0.345, ActualMean=19223.66, PredMean=17044.96


Random_Forest: MAE=13369.62, RMSE=21602.39, R2=0.128, ActualMean=19223.66, PredMean=17287.38


...

Random_Forest: MAE=8541.19, RMSE=13271.76, R2=-0.285, ActualMean=13210.28, PredMean=6660.13


Gradient_Boosting: MAE=8455.34, RMSE=13293.94, R2=-0.290, ActualMean=13210.28, PredMean=6258.76
SELECTED SEVERITY MODEL: Gamma_GLM



SEVERITY MODEL: Personal Property
Positive claims: 743
Gamma_GLM: MAE=539.71, RMSE=814.38, R2=0.361, ActualMean=1097.59, PredMean=1061.08


Random_Forest: MAE=554.33, RMSE=836.57, R2=0.326, ActualMean=1097.59, PredMean=1079.23


Gradient_Boosting: MAE=567.40, RMSE=878.23, R2=0.257, ActualMean=1097.59, PredMean=981.52
SELECTED SEVERITY MODEL: Gamma_GLM


In [24]:
# 11. SCORE THE OFFICIAL HOLDOUT DATA


In [25]:
holdout_scored_parts = []

for coverage in coverages:
    cov_holdout = holdout_df[
        holdout_df["coverage"] == coverage
    ].copy()

    if cov_holdout.empty:
        continue

    freq_model = frequency_models[coverage]["model"]
    sev_model = severity_models[coverage]["model"]

    cov_holdout["predicted_frequency"] = (
        freq_model.predict_proba(
            cov_holdout[features]
        )[:, 1]
    )

    cov_holdout["predicted_severity"] = np.maximum(
        sev_model.predict(cov_holdout[features]),
        0
    )

    # Apply known contractual coverage limits where available.
    if coverage in COVERAGE_LIMITS:
        cov_holdout["predicted_severity"] = (
            cov_holdout["predicted_severity"]
            .clip(upper=COVERAGE_LIMITS[coverage])
        )

    cov_holdout["predicted_expected_loss"] = (
        cov_holdout["predicted_frequency"]
        * cov_holdout["predicted_severity"]
    )

    cov_holdout["frequency_model"] = (
        frequency_models[coverage]["name"]
    )
    cov_holdout["severity_model"] = (
        severity_models[coverage]["name"]
    )

    # Liability credibility adjustment: shrink model predictions toward the
    # Liability development-sample aggregate where claim volume is limited.
    # This is applied BEFORE looking at holdout losses and uses training data only.
    if str(coverage).lower() == "liability":
        li_train = train_df[train_df["coverage"] == coverage].copy()
        li_claims = li_train.loc[li_train["amount"] > 0, "amount"]
        li_claim_count = int(li_train["claim_count"].sum())
        li_freq_base = li_claim_count / len(li_train)
        li_sev_mean = li_claims.mean() if len(li_claims) else np.nan
        li_sev_cv = (li_claims.std(ddof=1) / li_sev_mean) if len(li_claims) > 1 and li_sev_mean > 0 else np.nan
        z_freq, n0_freq = limited_fluctuation_credibility(li_claim_count, cv=None)
        z_sev, n0_sev = limited_fluctuation_credibility(len(li_claims), cv=li_sev_cv)
        cov_holdout["credibility_z_frequency"] = z_freq
        cov_holdout["credibility_z_severity"] = z_sev
        cov_holdout["predicted_frequency_raw"] = cov_holdout["predicted_frequency"]
        cov_holdout["predicted_severity_raw"] = cov_holdout["predicted_severity"]
        cov_holdout["predicted_frequency"] = z_freq * cov_holdout["predicted_frequency"] + (1-z_freq) * li_freq_base
        cov_holdout["predicted_severity"] = z_sev * cov_holdout["predicted_severity"] + (1-z_sev) * li_sev_mean
        cov_holdout["credibility_frequency_base"] = li_freq_base
        cov_holdout["credibility_severity_base"] = li_sev_mean
        cov_holdout["credibility_n0_frequency"] = n0_freq
        cov_holdout["credibility_n0_severity"] = n0_sev
        cov_holdout["predicted_expected_loss"] = cov_holdout["predicted_frequency"] * cov_holdout["predicted_severity"]

    holdout_scored_parts.append(cov_holdout)

holdout_scored = pd.concat(
    holdout_scored_parts,
    ignore_index=True
)


In [26]:
# 12. OFFICIAL HOLDOUT VALIDATION


In [27]:
holdout_model_summary = []

for coverage, g in holdout_scored.groupby("coverage"):
    freq = frequency_metrics(
        g["has_claim"],
        g["predicted_frequency"],
        threshold=0.50
    )

    actual_positive = g[g["amount"] > 0].copy()

    if not actual_positive.empty:
        sev = severity_metrics(
            actual_positive["amount"],
            actual_positive["predicted_severity"]
        )
    else:
        sev = {
            "MAE": np.nan,
            "RMSE": np.nan,
            "R2": np.nan,
            "Actual_Mean": np.nan,
            "Predicted_Mean": np.nan,
        }

    holdout_model_summary.append({
        "Coverage": coverage,
        "Records": len(g),
        "Claims": int(g["has_claim"].sum()),
        "Actual_Loss": g["amount"].sum(),
        "Predicted_Expected_Loss": (
            g["predicted_expected_loss"].sum()
        ),
        "Actual_Pure_Premium": g["amount"].mean(),
        "Predicted_Pure_Premium": (
            g["predicted_expected_loss"].mean()
        ),
        "Loss_Ratio_Predicted_to_Actual": (
            g["predicted_expected_loss"].sum()
            / g["amount"].sum()
            if g["amount"].sum() > 0 else np.nan
        ),
        "Frequency_ROC_AUC": freq["ROC_AUC"],
        "Frequency_PR_AUC": freq["PR_AUC"],
        "Frequency_Brier": freq["Brier"],
        "Observed_Frequency": freq["Observed_Frequency"],
        "Mean_Predicted_Frequency": (
            freq["Mean_Predicted_Probability"]
        ),
        "Severity_MAE_on_Claims": sev["MAE"],
        "Severity_RMSE_on_Claims": sev["RMSE"],
        "Severity_R2_on_Claims": sev["R2"],
    })

holdout_model_summary = pd.DataFrame(
    holdout_model_summary
)

print("\nOFFICIAL HOLDOUT VALIDATION")
print(holdout_model_summary)



OFFICIAL HOLDOUT VALIDATION
                    Coverage  Records  Claims    Actual_Loss  \
0  Additional Living Expense     2001      96  580920.695392   
1              Guest Medical     2001      46  681367.090543   
2                  Liability     2001      18  191302.115192   
3          Personal Property     2001     179  188373.614365   

   Predicted_Expected_Loss  Actual_Pure_Premium  Predicted_Pure_Premium  \
0            573530.532215           290.315190              286.621955   
1            770334.071787           340.513289              384.974549   
2            130563.535842            95.603256               65.249143   
3            201982.783777            94.139737              100.940921   

   Loss_Ratio_Predicted_to_Actual  Frequency_ROC_AUC  Frequency_PR_AUC  \
0                        0.987279           0.591076          0.076462   
1                        1.130571           0.565356          0.028859   
2                        0.682499           0.684905

In [28]:
# 13. DEVELOPMENT-SAMPLE RATING BASES AND RELATIVITIES

print("\nMETRIC DEFINITIONS")
print("Frequency metrics above are official HOLDOUT metrics.")
print("Severity metrics above are evaluated only on HOLDOUT exposures with observed positive loss; they are NOT training metrics.")


METRIC DEFINITIONS
Frequency metrics above are official HOLDOUT metrics.
Severity metrics above are evaluated only on HOLDOUT exposures with observed positive loss; they are NOT training metrics.


## Risk-tier interpretation

Interpretation is descriptive: a higher observed pure premium means higher average loss per exposure in that tier, but it does not by itself establish causality. We also examine frequency and severity separately because a tier can have lower claim frequency but higher claim severity.

For pricing, tier relativities are constructed from the **development sample only** after the predictive models are fitted. The official holdout is used only to test how well those rates generalize.


In [29]:
 # Rating bases are developed from the official training/development sample only.
# The official holdout is reserved for final validation and is not used to construct rates.

# Score the development sample with the final models.
train_scored_parts = []
for coverage in coverages:
    cov_train = train_df[train_df["coverage"] == coverage].copy()
    freq_model = frequency_models[coverage]["model"]
    sev_model = severity_models[coverage]["model"]

    cov_train["predicted_frequency"] = freq_model.predict_proba(cov_train[features])[:, 1]
    cov_train["predicted_severity"] = np.maximum(sev_model.predict(cov_train[features]), 0)

    if coverage in COVERAGE_LIMITS:
        cov_train["predicted_severity"] = cov_train["predicted_severity"].clip(upper=COVERAGE_LIMITS[coverage])

    cov_train["predicted_expected_loss"] = (
        cov_train["predicted_frequency"] * cov_train["predicted_severity"]
    )
    train_scored_parts.append(cov_train)

train_scored = pd.concat(train_scored_parts, ignore_index=True)

# Tier base pure premiums are normalized within each coverage.
# Policy relativity is the model expected loss divided by the relevant tier base.
if "risk_tier" in train_scored.columns:
    tier_base_rates = (
        train_scored.groupby(["coverage", "risk_tier"], as_index=False)
        .agg(
            records=("student_id", "size"),
            observed_pure_premium=("amount", "mean"),
            predicted_frequency=("predicted_frequency", "mean"),
            predicted_severity=("predicted_severity", "mean"),
            tier_base_pure_premium=("predicted_expected_loss", "mean"),
        )
    )
    train_scored = train_scored.merge(
        tier_base_rates[["coverage", "risk_tier", "tier_base_pure_premium"]],
        on=["coverage", "risk_tier"], how="left"
    )
else:
    tier_base_rates = (
        train_scored.groupby("coverage", as_index=False)
        .agg(
            records=("student_id", "size"),
            observed_pure_premium=("amount", "mean"),
            predicted_frequency=("predicted_frequency", "mean"),
            predicted_severity=("predicted_severity", "mean"),
            tier_base_pure_premium=("predicted_expected_loss", "mean"),
        )
    )
    train_scored = train_scored.merge(
        tier_base_rates[["coverage", "tier_base_pure_premium"]],
        on="coverage", how="left"
    )

train_scored["policy_relativity"] = np.where(
    train_scored["tier_base_pure_premium"] > 0,
    train_scored["predicted_expected_loss"] / train_scored["tier_base_pure_premium"],
    1.0,
)

print("\nDEVELOPMENT-SAMPLE RATING BASES")
print(tier_base_rates)



DEVELOPMENT-SAMPLE RATING BASES
                     coverage risk_tier  records  observed_pure_premium  \
0   Additional Living Expense         1     3570             240.791409   
1   Additional Living Expense         2     1797             299.612367   
2   Additional Living Expense         3     2632             354.243024   
3               Guest Medical         1     3570             348.243393   
4               Guest Medical         2     1797             448.651236   
5               Guest Medical         3     2632             433.028048   
6                   Liability         1     3570              68.572877   
7                   Liability         2     1797              49.267041   
8                   Liability         3     2632              80.839912   
9           Personal Property         1     3570             100.005706   
10          Personal Property         2     1797             107.533462   
11          Personal Property         3     2632              97.57

In [30]:
# 14. VARIABLE-LEVEL RELATIVITY EXAMPLES


In [31]:
#
# These summaries are for explanation/business communication.
#
# Relativity = segment expected loss / relevant coverage base rate

segment_relativity_tables = []

segment_vars = [
    c for c in [
        "sprinklered",
        "greek",
        "off_campus",
        "risk_tier",
        "class",
        "study",
    ]
    if c in holdout_scored.columns
]

for segment_var in segment_vars:
    temp = (
        holdout_scored
        .groupby(["coverage", segment_var], as_index=False)
        .agg(
            records=("student_id", "size"),
            observed_frequency=("has_claim", "mean"),
            predicted_frequency=(
                "predicted_frequency", "mean"
            ),
            observed_average_loss=("amount", "mean"),
            model_expected_loss=(
                "predicted_expected_loss", "mean"
            ),
        )
    )

    coverage_base = (
        holdout_scored.groupby("coverage")[
            "predicted_expected_loss"
        ]
        .mean()
        .rename("coverage_base_expected_loss")
        .reset_index()
    )

    temp = temp.merge(
        coverage_base,
        on="coverage",
        how="left"
    )

    temp["model_relativity"] = (
        temp["model_expected_loss"]
        / temp["coverage_base_expected_loss"]
    )

    temp.insert(0, "segment_variable", segment_var)
    temp = temp.rename(
        columns={segment_var: "segment"}
    )

    segment_relativity_tables.append(temp)

segment_relativities = (
    pd.concat(segment_relativity_tables, ignore_index=True)
    if segment_relativity_tables
    else pd.DataFrame()
)


In [32]:
# 15. TECHNICAL PREMIUM


In [33]:
# 15. TECHNICAL PREMIUM
#
# Illustrative formula:
#   Technical Premium = (Expected Loss + Fixed Expense)
#                      / (1 - Variable Expense Ratio - Profit/Contingency Ratio)
#
# Interpretation:
#   Expected Loss = modeled pure premium before expenses/profit.
#   Fixed Expense = dollar expense per coverage exposure. Set to $0 here because
#                   the case does not provide a fixed expense amount.
#   Variable Expense Ratio = portion of premium allocated to variable expenses.
#   Profit/Contingency Ratio = portion of premium reserved for underwriting profit
#                              and contingency/risk margin.
#
# Illustrative assumptions used here: 20% variable expenses, 10% profit/contingency,
# and $0 fixed expense. These are NOT ABG-specific case facts and must be replaced
# by ABG expense/profit indications before implementation.
#
# With these assumptions: denominator = 1 - 0.20 - 0.10 = 0.70, so the technical
# premium is Expected Loss / 0.70. The implied expected-loss ratio is therefore 70%,
# variable expense ratio is 20%, and profit/contingency margin is 10%.

denominator = 1 - VARIABLE_EXPENSE_RATIO - PROFIT_AND_CONTINGENCY_RATIO
if denominator <= 0:
    raise ValueError("Variable expense + profit ratios must be less than 100%.")

holdout_scored["technical_premium"] = (
    (holdout_scored["predicted_expected_loss"] + FIXED_EXPENSE_PER_COVERAGE) / denominator
)
holdout_scored["technical_premium"] = holdout_scored["technical_premium"].clip(lower=MIN_PREMIUM)
if MAX_PREMIUM is not None:
    holdout_scored["technical_premium"] = holdout_scored["technical_premium"].clip(upper=MAX_PREMIUM)

pricing_assumptions = pd.DataFrame({
    "Assumption": ["Fixed expense per coverage", "Variable expense ratio", "Profit/contingency ratio", "Implied expected-loss ratio"],
    "Value": [FIXED_EXPENSE_PER_COVERAGE, VARIABLE_EXPENSE_RATIO, PROFIT_AND_CONTINGENCY_RATIO, denominator],
    "Status": ["Illustrative", "Illustrative", "Illustrative", "Derived"]
})
print("\nTECHNICAL PREMIUM ASSUMPTIONS")
print(pricing_assumptions)



TECHNICAL PREMIUM ASSUMPTIONS
                    Assumption  Value        Status
0   Fixed expense per coverage    0.0  Illustrative
1       Variable expense ratio    0.2  Illustrative
2     Profit/contingency ratio    0.1  Illustrative
3  Implied expected-loss ratio    0.7       Derived


In [34]:
# 16. POLICY-LEVEL TOTAL PREMIUM


In [35]:
#
# Each policyholder receives all four coverages.
# Sum the stand-alone technical premium across coverages.

policy_level_premium = (
    holdout_scored
    .groupby("student_id", as_index=False)
    .agg(
        predicted_total_expected_loss=(
            "predicted_expected_loss", "sum"
        ),
        total_technical_premium=(
            "technical_premium", "sum"
        ),
        actual_total_loss=("amount", "sum"),
    )
)

# Number of priced coverages per student
coverage_counts = (
    holdout_scored.groupby("student_id")["coverage"]
    .nunique()
    .rename("number_of_coverages")
    .reset_index()
)

policy_level_premium = policy_level_premium.merge(
    coverage_counts,
    on="student_id",
    how="left"
)


In [36]:
# 17. PREMIUM / LOSS RECONCILIATION


In [37]:
pricing_reconciliation = (
    holdout_scored.groupby("coverage", as_index=False)
    .agg(
        records=("student_id", "size"),
        actual_losses=("amount", "sum"),
        predicted_expected_losses=(
            "predicted_expected_loss", "sum"
        ),
        technical_premium=(
            "technical_premium", "sum"
        ),
    )
)

pricing_reconciliation[
    "actual_loss_ratio_on_technical_premium"
] = np.where(
    pricing_reconciliation["technical_premium"] > 0,
    pricing_reconciliation["actual_losses"]
    / pricing_reconciliation["technical_premium"],
    np.nan,
)

pricing_reconciliation[
    "predicted_loss_ratio_on_technical_premium"
] = np.where(
    pricing_reconciliation["technical_premium"] > 0,
    pricing_reconciliation["predicted_expected_losses"]
    / pricing_reconciliation["technical_premium"],
    np.nan,
)


In [38]:
# 18. FAIRNESS / MONITORING TABLE


In [39]:
# 18. FAIRNESS / SEGMENT MONITORING WITH CONFIDENCE INTERVALS

FAIRNESS_MIN_GROUP_SIZE = 100
CI_BOOTSTRAPS = 2000
fairness_frames = []
for var in ["gender", "greek"]:
    if var in holdout_scored.columns:
        rows = []
        for (coverage, group), g in holdout_scored.groupby(["coverage", var], dropna=False):
            n = len(g)
            claims = int(g["has_claim"].sum())
            freq_lo, freq_hi = wilson_interval(claims, n)
            pp_lo, pp_hi = bootstrap_mean_ci(g["amount"].values, CI_BOOTSTRAPS, seed=RANDOM_STATE)
            pred_pp_lo, pred_pp_hi = bootstrap_mean_ci(g["predicted_expected_loss"].values, CI_BOOTSTRAPS, seed=RANDOM_STATE+1)
            rows.append({
                "monitoring_variable": var, "coverage": coverage, "group": group,
                "records": n, "claims": claims,
                "observed_frequency": g["has_claim"].mean(),
                "observed_frequency_ci_low": freq_lo,
                "observed_frequency_ci_high": freq_hi,
                "observed_pure_premium": g["amount"].mean(),
                "observed_pure_premium_ci_low": pp_lo,
                "observed_pure_premium_ci_high": pp_hi,
                "predicted_pure_premium": g["predicted_expected_loss"].mean(),
                "predicted_pure_premium_ci_low": pred_pp_lo,
                "predicted_pure_premium_ci_high": pred_pp_hi,
                "average_technical_premium": g["technical_premium"].mean(),
                "small_group_flag": n < FAIRNESS_MIN_GROUP_SIZE,
            })
        fairness_frames.append(pd.DataFrame(rows))

fairness_monitoring = pd.concat(fairness_frames, ignore_index=True) if fairness_frames else pd.DataFrame()

# Segment-level confidence intervals for key non-demographic underwriting factors.
segment_frames = []
for var in ["risk_tier", "off_campus", "sprinklered"]:
    if var in holdout_scored.columns:
        for (coverage, group), g in holdout_scored.groupby(["coverage", var], dropna=False):
            freq_lo, freq_hi = wilson_interval(int(g["has_claim"].sum()), len(g))
            pp_lo, pp_hi = bootstrap_mean_ci(g["amount"].values, CI_BOOTSTRAPS, seed=RANDOM_STATE+2)
            segment_frames.append({
                "segment_variable": var, "coverage": coverage, "group": group,
                "records": len(g), "claims": int(g["has_claim"].sum()),
                "observed_frequency": g["has_claim"].mean(),
                "frequency_ci_low": freq_lo, "frequency_ci_high": freq_hi,
                "observed_pure_premium": g["amount"].mean(),
                "pure_premium_ci_low": pp_lo, "pure_premium_ci_high": pp_hi,
                "predicted_pure_premium": g["predicted_expected_loss"].mean(),
            })
segment_confidence_intervals = pd.DataFrame(segment_frames)

print("\nFAIRNESS MONITORING WITH 95% CONFIDENCE INTERVALS")
print(fairness_monitoring)
print("\nSEGMENT COMPARISONS WITH 95% CONFIDENCE INTERVALS")
print(segment_confidence_intervals)



FAIRNESS MONITORING WITH 95% CONFIDENCE INTERVALS
   monitoring_variable                   coverage            group  records  \
0               gender  Additional Living Expense  Did not respond       54   
1               gender  Additional Living Expense           Female      894   
2               gender  Additional Living Expense             Male      906   
3               gender  Additional Living Expense            Other      147   
4               gender              Guest Medical  Did not respond       54   
5               gender              Guest Medical           Female      894   
6               gender              Guest Medical             Male      906   
7               gender              Guest Medical            Other      147   
8               gender                  Liability  Did not respond       54   
9               gender                  Liability           Female      894   
10              gender                  Liability             Male      906   
1

## Final Pricing Framework — Development-Based Rating Bases

The model is used to generate policy-level risk indications, but the **tier base rates are constructed from development data only**. The official holdout remains a validation sample.

For each coverage and tier:

- Base pure premium = mean development predicted expected loss
- Policy relativity = policy predicted expected loss ÷ applicable tier base pure premium
- Technical premium = (base pure premium × policy relativity + fixed expense) ÷ (1 − variable expense − profit/contingency)

For Liability, the final policy indication uses credibility-adjusted frequency and severity before calculating expected loss.

In [40]:
# 23. FINAL BUSINESS PRICING — ALL POLICIES
#
# The final pricing view scores the complete portfolio using models fitted only to
# the development sample. Holdout observations are not used to fit models or rates.

all_pricing = pricing_df.copy()

# Start with model predictions from development-fitted models.
final_parts = []
for coverage in coverages:
    g = all_pricing[all_pricing["coverage"] == coverage].copy()
    freq_model = frequency_models[coverage]["model"]
    sev_model = severity_models[coverage]["model"]

    g["model_frequency_raw"] = freq_model.predict_proba(g[features])[:, 1]
    g["model_severity_raw"] = np.maximum(sev_model.predict(g[features]), 0)

    if coverage in COVERAGE_LIMITS:
        g["model_severity_raw"] = g["model_severity_raw"].clip(upper=COVERAGE_LIMITS[coverage])

    g["pricing_frequency"] = g["model_frequency_raw"]
    g["pricing_severity"] = g["model_severity_raw"]

    # Apply Liability credibility using development data only.
    if str(coverage).lower() == "liability":
        li_train = train_df[train_df["coverage"] == coverage].copy()
        li_claims = li_train.loc[li_train["amount"] > 0, "amount"]
        li_claim_count = int(li_train["claim_count"].sum())
        li_freq_base = li_claim_count / len(li_train)
        li_sev_mean = li_claims.mean()
        li_sev_cv = li_claims.std(ddof=1) / li_sev_mean
        z_freq, n0_freq = limited_fluctuation_credibility(li_claim_count, cv=None)
        z_sev, n0_sev = limited_fluctuation_credibility(len(li_claims), cv=li_sev_cv)

        g["credibility_z_frequency"] = z_freq
        g["credibility_z_severity"] = z_sev
        g["credibility_frequency_base"] = li_freq_base
        g["credibility_severity_base"] = li_sev_mean
        g["pricing_frequency"] = z_freq * g["model_frequency_raw"] + (1 - z_freq) * li_freq_base
        g["pricing_severity"] = z_sev * g["model_severity_raw"] + (1 - z_sev) * li_sev_mean
    else:
        g["credibility_z_frequency"] = 1.0
        g["credibility_z_severity"] = 1.0
        g["credibility_frequency_base"] = np.nan
        g["credibility_severity_base"] = np.nan

    g["pricing_expected_loss"] = g["pricing_frequency"] * g["pricing_severity"]
    final_parts.append(g)

final_pricing = pd.concat(final_parts, ignore_index=True)

# Merge development-only tier bases.
tier_base_lookup = tier_base_rates[["coverage", "risk_tier", "tier_base_pure_premium"]].copy()
final_pricing = final_pricing.merge(
    tier_base_lookup, on=["coverage", "risk_tier"], how="left", validate="many_to_one"
)

final_pricing["pricing_relativity"] = np.where(
    final_pricing["tier_base_pure_premium"] > 0,
    final_pricing["pricing_expected_loss"] / final_pricing["tier_base_pure_premium"],
    1.0,
)

final_pricing["technical_premium"] = (
    final_pricing["pricing_expected_loss"] + FIXED_EXPENSE_PER_COVERAGE
) / denominator

final_pricing["technical_premium"] = final_pricing["technical_premium"].clip(lower=MIN_PREMIUM)
if MAX_PREMIUM is not None:
    final_pricing["technical_premium"] = final_pricing["technical_premium"].clip(upper=MAX_PREMIUM)

final_pricing_summary = (
    final_pricing.groupby(["coverage", "risk_tier"], as_index=False)
    .agg(
        records=("student_id", "size"),
        average_pricing_frequency=("pricing_frequency", "mean"),
        average_pricing_severity=("pricing_severity", "mean"),
        average_expected_loss=("pricing_expected_loss", "mean"),
        tier_base_pure_premium=("tier_base_pure_premium", "first"),
        average_relativity=("pricing_relativity", "mean"),
        average_technical_premium=("technical_premium", "mean"),
    )
)

print("\nFINAL PRICING SUMMARY")
print(final_pricing_summary.round(4))


FINAL PRICING SUMMARY
                     coverage risk_tier  records  average_pricing_frequency  \
0   Additional Living Expense         1     4512                     0.0532   
1   Additional Living Expense         2     2209                     0.0519   
2   Additional Living Expense         3     3279                     0.0512   
3               Guest Medical         1     4512                     0.0228   
4               Guest Medical         2     2209                     0.0247   
5               Guest Medical         3     3279                     0.0213   
6                   Liability         1     4512                     0.0084   
7                   Liability         2     2209                     0.0088   
8                   Liability         3     3279                     0.0084   
9           Personal Property         1     4512                     0.1030   
10          Personal Property         2     2209                     0.0962   
11          Personal Property

## Proposed Coverage-Specific Technical Indications

The following table converts the development-only tier base pure premiums into illustrative technical premiums using the stated 20% variable expense and 10% profit/contingency assumptions. These assumptions must be replaced by ABG's approved expense and profit indications before implementation.

In [41]:
# 24. COVERAGE-SPECIFIC TIER TECHNICAL INDICATIONS
pricing_structure = tier_base_rates.copy()
pricing_structure["technical_premium"] = (
    pricing_structure["tier_base_pure_premium"] + FIXED_EXPENSE_PER_COVERAGE
) / denominator
pricing_structure = pricing_structure.sort_values(["coverage", "risk_tier"])

print("\nCOVERAGE-SPECIFIC TIER PRICING STRUCTURE")
print(pricing_structure[["coverage", "risk_tier", "tier_base_pure_premium", "technical_premium"]].round(2))


COVERAGE-SPECIFIC TIER PRICING STRUCTURE
                     coverage risk_tier  tier_base_pure_premium  \
0   Additional Living Expense         1                  248.55   
1   Additional Living Expense         2                  277.74   
2   Additional Living Expense         3                  306.86   
3               Guest Medical         1                  330.60   
4               Guest Medical         2                  420.54   
5               Guest Medical         3                  396.45   
6                   Liability         1                   58.00   
7                   Liability         2                   60.13   
8                   Liability         3                   71.32   
9           Personal Property         1                   97.23   
10          Personal Property         2                  105.69   
11          Personal Property         3                   91.09   

    technical_premium  
0              355.07  
1              396.77  
2             

## New Business vs. Renewal Pricing

**New business:** use the predictive model, development-based tier base rate, approved rating factors, and the applicable credibility treatment. No customer-specific prior ABG experience is assumed beyond the information legitimately available for the new risk.

**Renewal:** retain the model indication, then introduce ABG-specific observed experience gradually through credibility. This avoids allowing one or two claims to produce disproportionate renewal changes.

Conceptually: `Renewal indication = Z × credible observed experience + (1 − Z) × model/benchmark indication`.

The notebook does not use the current-period target losses to manufacture a renewal price; the function below is an implementation template for future renewal periods.

In [42]:
# 25. RENEWAL CREDIBILITY TEMPLATE
def renewal_indication(model_indication, observed_experience, Z):
    """Blend a model/benchmark indication with credible future renewal experience."""
    Z = float(np.clip(Z, 0.0, 1.0))
    return Z * observed_experience + (1.0 - Z) * model_indication

# Example only — replace with future ABG experience and approved credibility standards.
# renewal_example = renewal_indication(model_indication=500, observed_experience=650, Z=0.25)
# print(renewal_example)

renewal_framework = pd.DataFrame({
    "Business_Type": ["New Business", "Renewal — limited experience", "Renewal — credible experience"],
    "Primary_Indication": [
        "Predictive model + tier base rate",
        "Model + partial credibility on ABG experience",
        "Model + greater credibility on ABG experience",
    ],
    "Governance_Point": [
        "No ABG-specific prior experience weight",
        "Prevent excessive response to sparse experience",
        "Increase experience weight only as credibility develops",
    ],
})
print("\nNEW BUSINESS / RENEWAL FRAMEWORK")
print(renewal_framework)


NEW BUSINESS / RENEWAL FRAMEWORK
                   Business_Type  \
0                   New Business   
1   Renewal — limited experience   
2  Renewal — credible experience   

                              Primary_Indication  \
0              Predictive model + tier base rate   
1  Model + partial credibility on ABG experience   
2  Model + greater credibility on ABG experience   

                                    Governance_Point  
0            No ABG-specific prior experience weight  
1    Prevent excessive response to sparse experience  
2  Increase experience weight only as credibility...  


In [43]:
# 19. SIMPLE 1-3 YEAR PRO FORMA


In [44]:
# 19. ILLUSTRATIVE 1-3 YEAR SCENARIO ANALYSIS

PRO_FORMA_SCENARIOS = {
    "Lower_Growth": {"annual_policy_growth": 0.02, "annual_loss_trend": 0.03, "annual_premium_change": 0.02},
    "Base": {"annual_policy_growth": 0.05, "annual_loss_trend": 0.04, "annual_premium_change": 0.03},
    "Higher_Growth": {"annual_policy_growth": 0.08, "annual_loss_trend": 0.05, "annual_premium_change": 0.04},
}

base_policyholders = policy_level_premium["student_id"].nunique()
base_premium = policy_level_premium["total_technical_premium"].sum()
base_expected_loss = policy_level_premium["predicted_total_expected_loss"].sum()

pro_forma_rows = []
for scenario, assumptions in PRO_FORMA_SCENARIOS.items():
    for year in [1, 2, 3]:
        policyholders = base_policyholders * ((1 + assumptions["annual_policy_growth"]) ** year)
        premium = base_premium * ((1 + assumptions["annual_policy_growth"]) ** year) * ((1 + assumptions["annual_premium_change"]) ** year)
        expected_loss = base_expected_loss * ((1 + assumptions["annual_policy_growth"]) ** year) * ((1 + assumptions["annual_loss_trend"]) ** year)
        pro_forma_rows.append({
            "Scenario": scenario,
            "Year": year,
            "Projected_Policyholders": policyholders,
            "Projected_Premium": premium,
            "Projected_Expected_Loss": expected_loss,
            "Projected_Loss_Ratio": expected_loss / premium if premium > 0 else np.nan,
        })

pro_forma = pd.DataFrame(pro_forma_rows)
print("\nILLUSTRATIVE SCENARIO ANALYSIS")
print(pro_forma)



ILLUSTRATIVE SCENARIO ANALYSIS
        Scenario  Year  Projected_Policyholders  Projected_Premium  \
0   Lower_Growth     1              2041.020000       2.491626e+06   
1   Lower_Growth     2              2081.840400       2.592287e+06   
2   Lower_Growth     3              2123.477208       2.697016e+06   
3           Base     1              2101.050000       2.590055e+06   
4           Base     2              2206.102500       2.801144e+06   
5           Base     3              2316.407625       3.029438e+06   
6  Higher_Growth     1              2161.080000       2.689921e+06   
7  Higher_Growth     2              2333.966400       3.021319e+06   
8  Higher_Growth     3              2520.683712       3.393546e+06   

   Projected_Expected_Loss  Projected_Loss_Ratio  
0             1.761237e+06              0.706863  
1             1.850356e+06              0.713793  
2             1.943984e+06              0.720791  
3             1.830641e+06              0.706796  
4           

In [45]:
# 20A. HOLDOUT CALIBRATION DIAGNOSTICS
# Frequency calibration and pure-premium calibration by decile are evaluated
# only on the official holdout. These are diagnostic plots, not tuning inputs.
calibration_rows = []
for coverage, g in holdout_scored.groupby("coverage"):
    g = g.copy()
    g["freq_decile"] = pd.qcut(g["predicted_frequency"].rank(method="first"), 10, labels=False) + 1
    g["pp_decile"] = pd.qcut(g["predicted_expected_loss"].rank(method="first"), 10, labels=False) + 1
    for d, x in g.groupby("freq_decile"):
        calibration_rows.append({"coverage": coverage, "calibration_type": "Frequency", "decile": d,
                                 "mean_predicted": x["predicted_frequency"].mean(),
                                 "mean_observed": x["has_claim"].mean(), "records": len(x)})
    for d, x in g.groupby("pp_decile"):
        calibration_rows.append({"coverage": coverage, "calibration_type": "Pure Premium", "decile": d,
                                 "mean_predicted": x["predicted_expected_loss"].mean(),
                                 "mean_observed": x["amount"].mean(), "records": len(x)})

holdout_calibration = pd.DataFrame(calibration_rows)
print("\nHOLDOUT CALIBRATION")
print(holdout_calibration)



HOLDOUT CALIBRATION
                     coverage calibration_type  decile  mean_predicted  \
0   Additional Living Expense        Frequency       1        0.032124   
1   Additional Living Expense        Frequency       2        0.036122   
2   Additional Living Expense        Frequency       3        0.038572   
3   Additional Living Expense        Frequency       4        0.041197   
4   Additional Living Expense        Frequency       5        0.043891   
..                        ...              ...     ...             ...   
75          Personal Property     Pure Premium       6       72.810701   
76          Personal Property     Pure Premium       7       87.474368   
77          Personal Property     Pure Premium       8      110.872248   
78          Personal Property     Pure Premium       9      176.642620   
79          Personal Property     Pure Premium      10      328.886935   

    mean_observed  records  
0        0.044776      201  
1        0.030000      200  
2  

In [46]:
# 21. FINAL SANITY CHECKS


In [47]:
# 20. ADDITIONAL ACTUARIAL DIAGNOSTICS

# Claim-count diagnostic: the binary frequency target models probability of at least one claim.
# Because a small number of student-coverage exposures have multiple claims, inspect count behavior.
claim_count_summary = (
    train_df.groupby("coverage", as_index=False)
    .agg(
        exposures=("student_id", "size"),
        total_claims=("claim_count", "sum"),
        multi_claim_exposures=("claim_count", lambda s: int((s > 1).sum())),
        max_claims_per_exposure=("claim_count", "max"),
        mean_claim_count=("claim_count", "mean"),
    )
)
claim_count_summary["claim_frequency_count_basis"] = (
    claim_count_summary["total_claims"] / claim_count_summary["exposures"]
)

# Holdout aggregate calibration by risk tier and coverage.
if "risk_tier" in holdout_scored.columns:
    holdout_tier_calibration = (
        holdout_scored.groupby(["coverage", "risk_tier"], as_index=False)
        .agg(
            records=("student_id", "size"),
            actual_pure_premium=("amount", "mean"),
            predicted_pure_premium=("predicted_expected_loss", "mean"),
            observed_frequency=("has_claim", "mean"),
            predicted_frequency=("predicted_frequency", "mean"),
        )
    )
    holdout_tier_calibration["prediction_to_actual"] = np.where(
        holdout_tier_calibration["actual_pure_premium"] != 0,
        holdout_tier_calibration["predicted_pure_premium"] / holdout_tier_calibration["actual_pure_premium"],
        np.nan,
    )
else:
    holdout_tier_calibration = pd.DataFrame()

# Flag low-claim-volume coverage results for credibility review.
LOW_CLAIM_HOLDOUT_THRESHOLD = 25
holdout_model_summary["low_claim_volume_flag"] = holdout_model_summary["Claims"] < LOW_CLAIM_HOLDOUT_THRESHOLD

print("\nCLAIM COUNT DIAGNOSTIC")
print(claim_count_summary)
print("\nHOLDOUT TIER CALIBRATION")
print(holdout_tier_calibration)
print("\nLOW CLAIM-VOLUME FLAGS")
print(holdout_model_summary[["Coverage", "Claims", "low_claim_volume_flag"]])


# Liability credibility summary, based only on development data.
li_train = train_df[train_df["coverage"].astype(str).str.lower() == "liability"].copy()
li_claims = li_train.loc[li_train["amount"] > 0, "amount"]
li_claim_count = int(li_train["claim_count"].sum())
li_freq = li_claim_count / len(li_train)
li_sev_mean = li_claims.mean()
li_cv = li_claims.std(ddof=1) / li_sev_mean
li_z_freq, li_n0_freq = limited_fluctuation_credibility(li_claim_count, cv=None)
li_z_sev, li_n0_sev = limited_fluctuation_credibility(len(li_claims), cv=li_cv)
liability_credibility = pd.DataFrame([{
    "coverage": "Liability", "development_exposures": len(li_train),
    "development_claims": li_claim_count, "severity_claims": len(li_claims),
    "development_frequency": li_freq, "development_mean_severity": li_sev_mean,
    "severity_cv": li_cv, "frequency_credibility_Z": li_z_freq,
    "severity_credibility_Z": li_z_sev, "frequency_full_credibility_claims": li_n0_freq,
    "severity_full_credibility_claims": li_n0_sev,
    "confidence": CREDIBILITY_CONFIDENCE, "relative_error": CREDIBILITY_RELATIVE_ERROR
}])
print("\nLIABILITY CREDIBILITY TREATMENT")
print(liability_credibility)



CLAIM COUNT DIAGNOSTIC
                    coverage  exposures  total_claims  multi_claim_exposures  \
0  Additional Living Expense       7999           427                     11   
1              Guest Medical       7999           187                      4   
2                  Liability       7999            67                      0   
3          Personal Property       7999           787                     41   

   max_claims_per_exposure  mean_claim_count  claim_frequency_count_basis  
0                        2          0.053382                     0.053382  
1                        2          0.023378                     0.023378  
2                        1          0.008376                     0.008376  
3                        3          0.098387                     0.098387  

HOLDOUT TIER CALIBRATION
                     coverage risk_tier  records  actual_pure_premium  \
0   Additional Living Expense         1      942           314.633818   
1   Additional Living E

In [48]:
print("\n" + "=" * 70)
print("FINAL SANITY CHECKS")
print("=" * 70)

print(
    "Holdout observed frequency:",
    holdout_scored["has_claim"].mean()
)
print(
    "Holdout average predicted frequency:",
    holdout_scored["predicted_frequency"].mean()
)

print(
    "Holdout actual total loss:",
    holdout_scored["amount"].sum()
)
print(
    "Holdout predicted total expected loss:",
    holdout_scored["predicted_expected_loss"].sum()
)

print(
    "Average technical premium per coverage record:",
    holdout_scored["technical_premium"].mean()
)
print(
    "Average total technical premium per policyholder:",
    policy_level_premium["total_technical_premium"].mean()
)

# Flag probability calibration problems.
observed_freq = holdout_scored["has_claim"].mean()
pred_freq = holdout_scored["predicted_frequency"].mean()

if observed_freq > 0:
    probability_ratio = pred_freq / observed_freq
    print(
        "Predicted / observed frequency ratio:",
        probability_ratio
    )
    if probability_ratio > 1.25 or probability_ratio < 0.80:
        print(
            "WARNING: Average predicted claim probability differs "
            "materially from observed holdout frequency."
        )

# Check expected loss calibration.
actual_loss = holdout_scored["amount"].sum()
pred_loss = holdout_scored[
    "predicted_expected_loss"
].sum()

if actual_loss > 0:
    loss_ratio = pred_loss / actual_loss
    print("Predicted / actual loss ratio:", loss_ratio)

    if loss_ratio > 1.25 or loss_ratio < 0.80:
        print(
            "WARNING: Aggregate predicted expected loss differs "
            "materially from actual holdout loss."
        )



FINAL SANITY CHECKS
Holdout observed frequency: 0.04235382308845577
Holdout average predicted frequency: 0.04461985497297358
Holdout actual total loss: 1641963.51549269
Holdout predicted total expected loss: 1676410.923620989
Average technical premium per coverage record: 299.20948875936824
Average total technical premium per policyholder: 1196.837955037473
Predicted / observed frequency ratio: 1.053502416530031
Predicted / actual loss ratio: 1.0209793992395517


In [49]:
# 22. SAVE OUTPUTS


In [50]:
# Core CSV outputs
claim_count_summary.to_csv(
    OUTPUT_DIR / "claim_count_diagnostic.csv", index=False
)
holdout_tier_calibration.to_csv(
    OUTPUT_DIR / "holdout_tier_calibration.csv", index=False
)

overall_summary.to_csv(
    OUTPUT_DIR / "overall_summary.csv",
    index=False
)
coverage_summary.to_csv(
    OUTPUT_DIR / "coverage_summary.csv",
    index=False
)

if not risk_tier_summary.empty:
    risk_tier_summary.to_csv(
        OUTPUT_DIR / "risk_tier_summary.csv",
        index=False
    )

frequency_metrics_df.to_csv(
    OUTPUT_DIR / "frequency_model_comparison.csv",
    index=False
)
severity_metrics_df.to_csv(
    OUTPUT_DIR / "severity_model_comparison.csv",
    index=False
)
lift_df.to_csv(
    OUTPUT_DIR / "frequency_lift_tables.csv",
    index=False
)
frequency_validation_df.to_csv(
    OUTPUT_DIR / "frequency_validation_predictions.csv",
    index=False
)
severity_validation_df.to_csv(
    OUTPUT_DIR / "severity_validation_predictions.csv",
    index=False
)
holdout_model_summary.to_csv(
    OUTPUT_DIR / "official_holdout_validation.csv",
    index=False
)
tier_base_rates.to_csv(
    OUTPUT_DIR / "tier_base_rates.csv",
    index=False
)
segment_relativities.to_csv(
    OUTPUT_DIR / "segment_relativities.csv",
    index=False
)
holdout_scored.to_csv(
    OUTPUT_DIR / "final_pricing_by_coverage.csv",
    index=False
)
policy_level_premium.to_csv(
    OUTPUT_DIR / "final_policyholder_premiums.csv",
    index=False
)
pricing_reconciliation.to_csv(
    OUTPUT_DIR / "pricing_reconciliation.csv",
    index=False
)
pro_forma.to_csv(
    OUTPUT_DIR / "pro_forma_1_to_3_years.csv",
    index=False
)

if not fairness_monitoring.empty:
    fairness_monitoring.to_csv(
        OUTPUT_DIR / "fairness_monitoring.csv",
        index=False
    )


# Excel workbook containing the main deliverables.
excel_path = OUTPUT_DIR / "ABG_Complete_Analysis_Results.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    overall_summary.to_excel(
        writer,
        sheet_name="Overall Summary",
        index=False
    )
    coverage_summary.to_excel(
        writer,
        sheet_name="Coverage Summary",
        index=False
    )

    if not risk_tier_summary.empty:
        risk_tier_summary.to_excel(
            writer,
            sheet_name="Risk Tier Summary",
            index=False
        )

    frequency_metrics_df.to_excel(
        writer,
        sheet_name="Frequency Models",
        index=False
    )
    severity_metrics_df.to_excel(
        writer,
        sheet_name="Severity Models",
        index=False
    )
    lift_df.to_excel(
        writer,
        sheet_name="Lift Tables",
        index=False
    )
    holdout_model_summary.to_excel(
        writer,
        sheet_name="Holdout Validation",
        index=False
    )
    tier_base_rates.to_excel(
        writer,
        sheet_name="Tier Base Rates",
        index=False
    )

    if not segment_relativities.empty:
        segment_relativities.to_excel(
            writer,
            sheet_name="Segment Relativities",
            index=False
        )

    holdout_scored.to_excel(
        writer,
        sheet_name="Coverage Pricing",
        index=False
    )
    policy_level_premium.to_excel(
        writer,
        sheet_name="Policy Premiums",
        index=False
    )
    pricing_reconciliation.to_excel(
        writer,
        sheet_name="Reconciliation",
        index=False
    )

    if not fairness_monitoring.empty:
        fairness_monitoring.to_excel(
            writer,
            sheet_name="Fairness Monitoring",
            index=False
        )

    claim_count_summary.to_excel(writer, sheet_name="Claim Count Diagnostic", index=False)
    holdout_tier_calibration.to_excel(writer, sheet_name="Holdout Tier Calibration", index=False)

    pro_forma.to_excel(
        writer,
        sheet_name="Pro Forma",
        index=False
    )

    loss_reconciliation.to_excel(writer, sheet_name="Loss Reconciliation", index=False)
    risk_tier_construction.to_excel(writer, sheet_name="Risk Tier Construction", index=False)
    liability_credibility.to_excel(writer, sheet_name="Liability Credibility", index=False)
    segment_confidence_intervals.to_excel(writer, sheet_name="Segment CIs", index=False)
    pricing_assumptions.to_excel(writer, sheet_name="Pricing Assumptions", index=False)

print("\n" + "=" * 70)
print("ANALYSIS COMPLETE")
print("=" * 70)
print(f"Outputs saved to: {OUTPUT_DIR.resolve()}")
print(f"Main Excel output: {excel_path.resolve()}")


loss_reconciliation.to_csv(OUTPUT_DIR / "loss_reconciliation.csv", index=False)
risk_tier_construction.to_csv(OUTPUT_DIR / "risk_tier_construction.csv", index=False)
liability_credibility.to_csv(OUTPUT_DIR / "liability_credibility.csv", index=False)
segment_confidence_intervals.to_csv(OUTPUT_DIR / "segment_confidence_intervals.csv", index=False)
pricing_assumptions.to_csv(OUTPUT_DIR / "pricing_assumptions.csv", index=False)

holdout_calibration.to_csv(OUTPUT_DIR / "holdout_calibration.csv", index=False)



ANALYSIS COMPLETE
Outputs saved to: C:\Users\acer\Documents\ABG RENTERS\abg_final_outputs
Main Excel output: C:\Users\acer\Documents\ABG RENTERS\abg_final_outputs\ABG_Complete_Analysis_Results.xlsx


In [51]:
# 26. FINAL EXPORTS
final_pricing.to_csv(OUTPUT_DIR / "final_policy_pricing_framework.csv", index=False)
final_pricing_summary.to_csv(OUTPUT_DIR / "final_pricing_summary.csv", index=False)
pricing_structure.to_csv(OUTPUT_DIR / "coverage_tier_pricing_structure.csv", index=False)
renewal_framework.to_csv(OUTPUT_DIR / "new_vs_renewal_framework.csv", index=False)

# Append final pricing sheets to the existing workbook.
excel_path = OUTPUT_DIR / "ABG_Final_Pricing_Strategy_Results.xlsx"
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    overall_summary.to_excel(writer, sheet_name="Overall Summary", index=False)
    coverage_summary.to_excel(writer, sheet_name="Coverage Summary", index=False)
    holdout_model_summary.to_excel(writer, sheet_name="Holdout Validation", index=False)
    tier_base_rates.to_excel(writer, sheet_name="Tier Base Rates", index=False)
    pricing_structure.to_excel(writer, sheet_name="Coverage Pricing Structure", index=False)
    final_pricing_summary.to_excel(writer, sheet_name="Final Pricing Summary", index=False)
    final_pricing.to_excel(writer, sheet_name="Final Policy Pricing", index=False)
    liability_credibility.to_excel(writer, sheet_name="Liability Credibility", index=False)
    holdout_tier_calibration.to_excel(writer, sheet_name="Holdout Tier Calibration", index=False)
    fairness_monitoring.to_excel(writer, sheet_name="Fairness Monitoring", index=False)
    segment_confidence_intervals.to_excel(writer, sheet_name="Segment CIs", index=False)
    renewal_framework.to_excel(writer, sheet_name="New vs Renewal", index=False)
    pro_forma.to_excel(writer, sheet_name="Pro Forma", index=False)
    pricing_assumptions.to_excel(writer, sheet_name="Pricing Assumptions", index=False)

print("\nFINAL OUTPUTS SAVED")
print("Output directory:", OUTPUT_DIR.resolve())
print("Final workbook:", excel_path.resolve())



FINAL OUTPUTS SAVED
Output directory: C:\Users\acer\Documents\ABG RENTERS\abg_final_outputs
Final workbook: C:\Users\acer\Documents\ABG RENTERS\abg_final_outputs\ABG_Final_Pricing_Strategy_Results.xlsx
